This notebook builds `data/disaster_<ID>_agg_info.csv` from the NetMob 2024 dataset and Dartmouth Flood Observatory / NASA Global Flood Archive polygons. NetMob access is granted on request through the World Bank Data Catalog and is not redistributable by the authors, so the aggregated outputs are included in `data/` and the later scripts in this package read those. `data/grid_to_exclude4859.xlsx` is also included; it lists the 43 Geohash-5 cells excluded from the Bandung aggregation. The notebook geocodes administrative boundaries through `osmnx` (`ox.geocode_to_gdf`), so it needs network access.

In [ ]:
import os
import glob
import warnings 
import numpy as np
import pandas as pd
from datetime import datetime
import geopandas as gpd
from tqdm import tqdm
import multiprocessing as mp
from multiprocessing import Pool, cpu_count
import osmnx as ox

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
# from geopy.distance import geodesic
import statsmodels.api as sm
import statsmodels.formula.api as smf
# from stargazer.stargazer import Stargazer, LineLocation, HTMLRenderer

import utils

# Set up the logging so that we can see progress
import logging
# logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
from datetime import datetime


In [ ]:
def setup_logging(log_filename='app', **kwargs):
    log_dir = kwargs.get('log_dir')
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file_path = os.path.join(log_dir, f"{timestamp}_{log_filename}.log")
    
    # Create handlers with explicit levels
    file_handler = logging.FileHandler(log_file_path, mode='w')
    file_handler.setLevel(logging.INFO)
    
    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    
    # Configure logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[file_handler, stream_handler]
    )
    
    # Get and set the root logger level
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    
    logging.info("Logging system initialized")


## Get disaster info

In [ ]:
country = ['Indonesia', 'India']
flood_archive = gpd.read_file("../data/FloodArchive_NASA/FloodArchive_region.shp")
flood_archive['began_year'] = flood_archive['BEGAN'].dt.year
flood_archive = flood_archive[flood_archive['began_year'].isin([2019, 2020]) & flood_archive['COUNTRY'].isin(country)]
flood_archive = flood_archive.set_crs(epsg=4326).sort_values(by=['COUNTRY', 'BEGAN'])

In [ ]:
# flood_archive[flood_archive['ID'].isin([4775, 4859])].explore()

In [ ]:
# Published disasters for this replication package: 4775 (Pune, India) and
# 4859 (West Java, Indonesia). The source analysis also scoped a third,
# overlapping West Java event (4866) alongside 4859, but only 4775 and 4859
# were carried through to publication (data/disaster_{4775,4859}_agg_info.csv);
# 4866 is dropped here. Both published IDs are produced by a single
# top-to-bottom run of this notebook -- 4775 by the India loop below, 4859 by
# the Indonesia loop plus its grid-exclusion correction -- with no manual
# re-run or hand-edited constant required.
flood_of_interest = flood_archive[flood_archive['ID'].isin([4775, 4859])].drop(columns=['GLIDENUMBE', 'AREA', 'OTHERCOUNT'])
flood_of_interest['admin'] = flood_of_interest['ID'].map({4775: 'Pune', 4859: 'West Java'})
flood_of_interest

admin_geo = ox.geocode_to_gdf(['Pune','West Java'])[['display_name', 'geometry']].reset_index(drop=True)
admin_geo['admin'] = ['Pune', 'West Java']
admin_geo = admin_geo.rename(columns={'geometry': 'admin_of_interest_geometry'})

flood_of_interest = flood_of_interest.merge(admin_geo, on='admin', how='left')

In [ ]:
flood_of_interest 

## Process PD

In [ ]:
def process_geohash(geo5, pd_df, in_disaster_agg, start_date, end_date):
    # Filter and sort the data for the given geohash
    geo5_dis_od = pd_df[pd_df['geohash_5'] == geo5].sort_values('local_date').copy().reset_index(drop=True)

    # Select the disaster happen before and after 30 days: earliest_date=start_date - 1mon, latest_date = end_date + 1mon
    start_minus_1mon = pd.to_datetime(str(start_date), format='%Y%m%d') - pd.DateOffset(months=1)
    # start_minus_1mon = pd.to_datetime(str(start_date), format='%Y%m%d') - pd.DateOffset(days=15)
    start_minus_1mon = int(start_minus_1mon.strftime('%Y%m%d'))
    end_plus_1mon = pd.to_datetime(str(end_date), format='%Y%m%d') + pd.DateOffset(months=1)
    # end_plus_1mon = pd.to_datetime(str(end_date), format='%Y%m%d') + pd.DateOffset(days=15)
    end_plus_1mon = int(end_plus_1mon.strftime('%Y%m%d'))
    geo5_dis_od = geo5_dis_od[(geo5_dis_od['local_date'] >= start_minus_1mon) & (geo5_dis_od['local_date'] <= end_plus_1mon)].copy().reset_index(drop=True)
    
    # Mark affected rows based on the date range
    geo5_dis_od['date_affected'] = geo5_dis_od.apply(lambda x: 1 if start_date <= x['local_date'] <= end_date else 0, axis=1)
    
    # Compute the baseline: average number of unique users before the disaster
    baseline = geo5_dis_od[(geo5_dis_od['local_date'] < start_date) & (geo5_dis_od['local_date'] >= start_minus_1mon)]['no_of_unique_users'].mean()
    
    # Compute the change of users relative to the baseline
    geo5_dis_od['change_of_users'] = geo5_dis_od.apply(lambda x: (x['no_of_unique_users'] - baseline) / baseline, axis=1)
    
    # Compute the 3-day moving average of the change of users
    geo5_dis_od['change_of_users_3d_avg'] = geo5_dis_od['change_of_users'].rolling(window=3, min_periods=1).mean()
    
    # # Apply geometry from in_disaster_agg
    # geo5_dis_od['geometry'] = in_disaster_agg[in_disaster_agg['geohash_5'] == geo5]['geometry'].values[0]
    
    return geo5_dis_od

In [ ]:
data_root = '../data/netmob24_final_data/PD/daily/gh5'
res_root = '../outputs'

### India

In [ ]:
pd_info = pd.read_csv(f'{data_root}/pd_day_gh5_in_2019.csv')
unique_geohash5 = pd_info.drop_duplicates(subset=['geohash_5'])[['geohash_5']].copy().reset_index(drop=True)

In [ ]:
unique_geohash5['geometry'] = unique_geohash5.apply(lambda x: utils.geohash_to_polygon(x['geohash_5']), axis=1)
unique_geohash5 = gpd.GeoDataFrame(unique_geohash5, crs='EPSG:4326')

In [ ]:
for idx, row in tqdm(flood_of_interest[flood_of_interest['ID'].isin([4775])].iterrows(), total=flood_of_interest.shape[0]):
    dis_no = row['ID']
    save_dir = os.path.join(res_root, str(dis_no))
    utils.ensure_path(save_dir)
    admin_geo = row['admin_of_interest_geometry']
    dis_geo = row['geometry']
    start_date = int(row['BEGAN'].strftime("%Y%m%d"))
    end_date = int(row['ENDED'].strftime("%Y%m%d"))

    admin_geohash = unique_geohash5[unique_geohash5['geometry'].apply(lambda x: admin_geo.intersects(x))]['geohash_5'].to_list()
    affected_geohash = unique_geohash5[
        (unique_geohash5['geohash_5'].isin(admin_geohash)) &
        (unique_geohash5['geometry'].apply(lambda x: dis_geo.intersects(x)))
    ]['geohash_5'].to_list()
    pool = mp.Pool(36)
    args = [(geo5, pd_info, None, start_date, end_date) for geo5 in admin_geohash]

    # Apply the function in parallel
    results = pool.starmap(process_geohash, args)

    # Close the pool and wait for the work to finish
    pool.close()
    pool.join()
    
    if len(results) > 0:
        # Combine the results into a single DataFrame
        dis_od_proc = pd.concat(results, ignore_index=True)
        
        # mark affected areas
        dis_od_proc['area_affected'] = dis_od_proc['geohash_5'].apply(lambda x: 1 if x in affected_geohash else 0)
        
        # Published flat in data/, not nested under outputs/<ID>/ -- this is the
        # final artifact for 4775 (no post-hoc exclusion needed for this event).
        save_path = os.path.join('../data', f'disaster_{dis_no}_agg_info.csv')
        dis_od_proc.to_csv(save_path, index=False)

In [ ]:
# len(admin_geohash), len(affected_geohash)

In [ ]:
dis_od_proc = pd.read_csv('../data/disaster_4775_agg_info.csv')

In [ ]:
dis_od_proc.head()

### Indonesia

In [ ]:
flood_of_interest[flood_of_interest['ID'].isin([4859])]

In [ ]:
pd_info_2019 = pd.read_csv(f'{data_root}/pd_day_gh5_id_2019.csv')
pd_info_2020 = pd.read_csv(f'{data_root}/pd_day_gh5_id_2020.csv')
pd_info = pd.concat([pd_info_2019, pd_info_2020])
unique_geohash5 = pd_info.drop_duplicates(subset=['geohash_5'])[['geohash_5']].copy().reset_index(drop=True)

unique_geohash5['geometry'] = unique_geohash5.apply(lambda x: utils.geohash_to_polygon(x['geohash_5']), axis=1)
unique_geohash5 = gpd.GeoDataFrame(unique_geohash5, crs='EPSG:4326')

In [ ]:
for idx, row in tqdm(flood_of_interest[flood_of_interest['ID'].isin([4859])].iterrows(), total=flood_of_interest.shape[0]):
    dis_no = row['ID']
    save_dir = os.path.join(res_root, str(dis_no))
    utils.ensure_path(save_dir)
    admin_geo = row['admin_of_interest_geometry']
    dis_geo = row['geometry']
    start_date = int(row['BEGAN'].strftime("%Y%m%d"))
    end_date = int(row['ENDED'].strftime("%Y%m%d"))

    admin_geohash = unique_geohash5[unique_geohash5['geometry'].apply(lambda x: admin_geo.intersects(x))]['geohash_5'].to_list()
    affected_geohash = unique_geohash5[
        (unique_geohash5['geohash_5'].isin(admin_geohash)) &
        (unique_geohash5['geometry'].apply(lambda x: dis_geo.intersects(x)))
    ]['geohash_5'].to_list()
    pool = mp.Pool(36)
    args = [(geo5, pd_info, None, start_date, end_date) for geo5 in admin_geohash]

    # Apply the function in parallel
    results = pool.starmap(process_geohash, args)

    # Close the pool and wait for the work to finish
    pool.close()
    pool.join()
    
    if len(results) > 0:
        # Combine the results into a single DataFrame
        dis_od_proc = pd.concat(results, ignore_index=True)
        
        # mark affected areas
        dis_od_proc['area_affected'] = dis_od_proc['geohash_5'].apply(lambda x: 1 if x in affected_geohash else 0)
        
        # Pre-exclusion write, flat in data/ to match cell 14. The
        # railway-corridor exclusion below reads dis_od_proc from memory, and
        # the final 4859 artifact is written after it.
        save_path = os.path.join('../data', f'disaster_{dis_no}_agg_info.csv')
        dis_od_proc.to_csv(save_path, index=False)

In [ ]:
dis_od_proc.sort_values(by='local_date')

In [ ]:
len(admin_geohash)

In [ ]:
len(affected_geohash)

In [ ]:
# grid_to_exclude4859.xlsx is a hand-curated list of geohash_5 cells to drop
# from the 4859 (West Java) aggregation -- an editorial correction, not
# something this notebook derives computationally.
gird_to_exclude = pd.read_excel('../data/grid_to_exclude4859.xlsx')['index'].to_list()
# `dis_od_proc` still holds the raw (pre-exclusion) 4859 aggregation written by
# the loop above. The private working notebook instead re-read this from a
# disaster_4859_agg_info_original.csv that no cell in that notebook ever wrote
# -- it only existed because someone manually copied/renamed the loop's output
# before overwriting it with the excluded version. Using the in-memory frame
# here removes that undocumented manual step, so one top-to-bottom run
# reproduces the published data/disaster_4859_agg_info.csv without it.
dis_4859 = dis_od_proc.copy()
print(dis_4859.shape)

In [ ]:
dis_4859 = dis_4859[~dis_4859['geohash_5'].isin(gird_to_exclude)]
dis_4859.shape

In [ ]:
dis_4859.to_csv('../data/disaster_4859_agg_info.csv', index=False)